In [16]:
import numpy as np
import pandas as pd

In [33]:
dtype_map = {
    "Task Id": "string",
    "FPOGV": "Int64", "FPOGID": "Int64",
    "LPMMV": "Int64", "RPMMV": "Int64",
    "BKID": "Int64",  "BKDUR": "float64", 
    "AOI": "string"
}
fixations_labeled = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_labeled_with_blocks.csv", dtype=dtype_map, low_memory=False)
gaze_labeled      = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_labeled_with_blocks.csv", dtype=dtype_map, low_memory=False)

In [34]:
def keep_on_off(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return only rows where df['label'] is 'ON' or 'OFF'.
    No phase-based mapping—just strict label filtering.
    """
    if "label" not in df.columns:
        raise KeyError("DataFrame has no 'label' column.")

    out = df.copy()
    out["label"] = out["label"].astype(str).str.strip().str.upper()
    out = out[out["label"].isin(["ON", "OFF"])].copy()

    # quick sanity check
    print("Kept rows:", len(out))
    print(out["label"].value_counts())

    return out

# Example usage
fixations_onoff = keep_on_off(fixations_labeled)
gaze_onoff      = keep_on_off(gaze_labeled)



fixations_onoff.to_csv("outputs/fixations_ON_OFF_only.csv", index=False)
gaze_onoff.to_csv("outputs/gaze_ON_OFF_only.csv", index=False)

Kept rows: 40862
label
ON     28295
OFF    12567
Name: count, dtype: int64
Kept rows: 1212581
label
ON     808805
OFF    403776
Name: count, dtype: int64


In [50]:
import numpy as np
import pandas as pd

# Columns we’ll use to define a "user/session" identity
GROUP_COLS = ["Video Name (folder)", "block_id"]

def _baseline_per_group(df: pd.DataFrame,
                        *,
                        label_col="label",
                        pupil_col="pupil_mean",
                        method="off_mean",
                        min_off_windows: int = 2) -> pd.DataFrame:
    """
    Return a dataframe with columns: GROUP_COLS + ['pupil_baseline'].
    method:
      - 'off_mean'  : mean pupil over OFF windows per group (preferred)
      - 'first_k'   : mean pupil over first K windows per group (set k via first_k param)
      - 'group_mean': mean over all windows per group (fallback if no OFF)
    """
    d = df.copy()
    # normalize labels
    if label_col in d.columns:
        d[label_col] = d[label_col].astype(str).str.upper().str.strip()

    # Option 1: OFF windows (preferred)
    if method == "off_mean":
        if label_col not in d.columns:
            raise ValueError("Need label column to use method='off_mean'.")
        off = d.loc[d[label_col] == "OFF", GROUP_COLS + [pupil_col]].dropna(subset=[pupil_col])
        base = (off.groupby(GROUP_COLS, dropna=False)[pupil_col]
                   .mean()
                   .rename("pupil_baseline")
                   .reset_index())
        # Fallback for groups with no OFF: use group mean
        have = set(map(tuple, base[GROUP_COLS].values))
        missing_mask = ~d[GROUP_COLS].apply(tuple, axis=1).isin(have)
        if missing_mask.any():
            grp_mean = (d.groupby(GROUP_COLS, dropna=False)[pupil_col]
                          .mean()
                          .rename("pupil_baseline")
                          .reset_index())
            base = pd.concat([base, grp_mean[~grp_mean[GROUP_COLS].apply(tuple, axis=1).isin(have)]],
                             ignore_index=True)

    elif method == "group_mean":
        base = (d.groupby(GROUP_COLS, dropna=False)[pupil_col]
                  .mean()
                  .rename("pupil_baseline")
                  .reset_index())

    else:
        raise ValueError("Unknown method. Use 'off_mean' or 'group_mean' (or extend).")

    return base


def add_pupil_mean_normalized(df: pd.DataFrame,
                              *,
                              pupil_col="pupil_mean",
                              label_col="label",
                              method="off_mean") -> pd.DataFrame:
    """
    Adds 3 new columns:
      - 'pupil_baseline' : per-(Video,User) baseline (OFF mean or group mean)
      - 'pupil_mean_delta' : pupil_mean - baseline (mm)
      - 'pupil_mean_norm'  : (pupil_mean - baseline) / baseline  (relative change)
    """
    d = df.copy()
    base = _baseline_per_group(d, label_col=label_col, pupil_col=pupil_col, method=method)
    d = d.merge(base, on=GROUP_COLS, how="left", validate="many_to_one")

    # compute deltas
    d["pupil_mean_delta"] = d[pupil_col] - d["pupil_baseline"]
    # guard against divide-by-zero or missing baseline
    denom = d["pupil_baseline"].replace(0, np.nan)
    d["pupil_mean_norm"] = d["pupil_mean_delta"] / denom

    return d

In [51]:
import numpy as np
import pandas as pd

TICK_HZ = 10_000_000  # TIMETICK(f=10000000)

# --- Build ONE shared anchor per (Video, User) across any number of tables ---
def build_shared_anchor(*dfs,
                        tick_col: str = "TIMETICK(f=10000000)",
                        group_cols=("Video Name (folder)", "User ID (file)")) -> pd.DataFrame:
    refs = []
    for d in dfs:
        tmp = d[list(group_cols) + [tick_col]].copy()
        tmp[tick_col] = pd.to_numeric(tmp[tick_col], errors="coerce")
        refs.append(tmp.groupby(list(group_cols), as_index=False)[tick_col].min()
                      .rename(columns={tick_col: "min_tick"}))
    ref = (pd.concat(refs, ignore_index=True)
             .groupby(list(group_cols), as_index=False)["min_tick"].min()
             .rename(columns={"min_tick": "anchor_tick"}))
    return ref

# 1) Window ids by TIMETICK (per Video+User), no task-start anchoring
def add_window_id_by_timetick(
    df: pd.DataFrame,
    *,
    win_sec: float,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)"),
    shared_ref: pd.DataFrame | None = None,
    out_col: str = "win_id",
) -> pd.DataFrame:
    out = df.copy()
    out[tick_col] = pd.to_numeric(out[tick_col], errors="coerce")
    win_ticks = int(round(win_sec * TICK_HZ))

    if shared_ref is not None:
        out = out.merge(shared_ref, on=list(group_cols), how="left", validate="many_to_one")
        anchor = out["anchor_tick"]
    else:
        anchor = out.groupby(list(group_cols))[tick_col].transform("min")

    out[out_col] = ((out[tick_col] - anchor) // win_ticks).astype("Int64")
    return out

# 1b) Refine window labels by majority-fraction (label denoising)
def refine_window_labels_by_fraction(
    df_with_win: pd.DataFrame,
    *,
    group_cols=("Video Name (folder)", "User ID (file)"),
    win_col="win_id",
    label_col="label",
    on_hi=0.80,
    off_lo=0.20,
    out_col="label",
) -> pd.DataFrame:
    d = df_with_win.copy()
    d[label_col] = d[label_col].astype(str).str.upper().str.strip()
    d = d[d[label_col].isin(["ON","OFF"])].copy()

    key = list(group_cols) + [win_col]
    d["_is_on"] = (d[label_col] == "ON").astype(float)

    frac = (d.groupby(key, dropna=False)["_is_on"]
              .mean()
              .rename("on_fraction")
              .reset_index())

    frac["label_refined"] = np.where(frac["on_fraction"] >= on_hi, "ON",
                              np.where(frac["on_fraction"] <= off_lo, "OFF", np.nan))

    d = d.merge(frac[key + ["label_refined","on_fraction"]],
                on=key, how="left", validate="many_to_one")

    d = d[~d["label_refined"].isna()].copy()
    d[out_col] = d["label_refined"]
    d.drop(columns=["_is_on","label_refined"], inplace=True)
    return d

# ---- small helpers for robust stats inside windows ----
def _mad(x):
    x = pd.Series(x, dtype="float64").dropna()
    if x.empty: return np.nan
    med = x.median()
    return (x - med).abs().median()

def _pupil_slope(tick, pupil):
    # robust slope via np.polyfit on valid points; time in seconds from first sample
    if tick is None:
        return np.nan
    tt = pd.Series(tick, dtype="float64")
    pp = pd.Series(pupil, dtype="float64")
    m = (~tt.isna()) & (~pp.isna())
    if m.sum() < 5:
        return np.nan
    tsec = (tt[m] - tt[m].iloc[0]) / TICK_HZ
    try:
        slope, _ = np.polyfit(tsec.values, pp[m].values, 1)
    except Exception:
        slope = np.nan
    return slope

def _circular_variance_deg(angles_deg):
    a = pd.Series(angles_deg, dtype="float64").dropna()
    if a.empty: return np.nan
    rad = np.deg2rad(a.values)
    C = np.mean(np.cos(rad))
    S = np.mean(np.sin(rad))
    R = np.sqrt(C*C + S*S)
    return 1 - R  # 0=tightly clustered, 1=uniform

def _gaze_entropy_2d(x, y, *, bins=8, min_valid=100):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < min_valid or y.size < min_valid:
        return np.nan
    x = np.clip(x, 0.0, 1.0)
    y = np.clip(y, 0.0, 1.0)
    H, _, _ = np.histogram2d(x, y, bins=bins, range=[[0,1],[0,1]], density=False)
    total = H.sum()
    if total <= 0:
        return np.nan
    p = (H / total).ravel()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

# 2) FIXATION features → cleaned + robust features
def window_features_fixations(fix_df: pd.DataFrame, *, win_sec: float) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]
    eps = 1e-6

    def _agg(w: pd.DataFrame) -> pd.Series:
        fix_count     = w["FPOGID"].nunique() if "FPOGID" in w else np.nan
        fix_mean_dur  = w["FPOGD"].mean()     if "FPOGD"  in w else np.nan
        fix_median_dur= w["FPOGD"].median()   if "FPOGD"  in w else np.nan
        fix_aoi_ratio = w["AOI"].notna().mean() if "AOI" in w else np.nan

        fix_efficiency = (fix_count / (abs(fix_mean_dur) + eps)) if pd.notna(fix_mean_dur) else np.nan
        fix_rate_per_s = (fix_count / win_sec) if pd.notna(fix_count) else np.nan

        return pd.Series({
            "fix_efficiency":  fix_efficiency,
            "fix_aoi_ratio":   fix_aoi_ratio,
            "fix_rate_per_s":  fix_rate_per_s,
            "fix_median_dur":  fix_median_dur,
        })

    return (fix_df.dropna(subset=["win_id"])
                 .groupby(keys, dropna=False)
                 .apply(_agg)
                 .reset_index())

# 3) GAZE features → cleaned + robust features
def window_features_gaze(gaze_df: pd.DataFrame, *, win_sec: float = 3.0) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]

    def _agg(w: pd.DataFrame) -> pd.Series:
        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)
        lv = (w["LPMMV"] == 1) if "LPMMV" in w else pd.Series(False, index=w.index)
        rv = (w["RPMMV"] == 1) if "RPMMV" in w else pd.Series(False, index=w.index)

        # --- dispersion (std-based) ---
        gx = w.loc[gv, "FPOGX"].std() if "FPOGX" in w else np.nan
        gy = w.loc[gv, "FPOGY"].std() if "FPOGY" in w else np.nan
        gaze_disp_total = (np.sqrt((0 if pd.isna(gx) else gx**2) + (0 if pd.isna(gy) else gy**2))
                           if (not pd.isna(gx) or not pd.isna(gy)) else np.nan)

        # --- pupil summary ---
        if "LPMM" in w and "RPMM" in w:
            l = w["LPMM"].where(lv)
            r = w["RPMM"].where(rv)
            pupil_row = pd.concat([l, r], axis=1).mean(axis=1, skipna=True)
            pupil_mean  = pupil_row.mean()
            pupil_var   = pupil_row.var(ddof=1)
            pupil_mad   = _mad(pupil_row)
            tick_col = "TIMETICK(f=10000000)"
            pupil_slope = _pupil_slope(w[tick_col] if tick_col in w else None, pupil_row)
        else:
            pupil_mean = pupil_var = pupil_mad = pupil_slope = np.nan

        # --- saccades ---
        saccade_mean_mag = w.loc[gv, "SACCADE_MAG"].mean() if "SACCADE_MAG" in w else np.nan
        if "SACCADE_DIR" in w:
            saccade_circ_var = _circular_variance_deg(w.loc[gv, "SACCADE_DIR"])
            saccade_dir_var  = w.loc[gv, "SACCADE_DIR"].var(ddof=1)
        else:
            saccade_circ_var = np.nan
            saccade_dir_var  = np.nan
        if pd.isna(saccade_dir_var):
            saccade_dir_var = 0.0  # keep old behavior

        # --- blinks & quality ---
        blink_count      = w.loc[gv, "BKID"].nunique() if "BKID" in w else np.nan
        blink_rate       = (blink_count / win_sec) if pd.notna(blink_count) else np.nan
        valid_gaze_ratio = gv.mean() if len(w) else np.nan


        return pd.Series({
            "gaze_dispersion_total": gaze_disp_total,
            "pupil_mean":            pupil_mean,
            "pupil_var":             0 if pd.isna(pupil_var) else pupil_var,
            "pupil_mad":             pupil_mad,
            "pupil_slope":           pupil_slope,
            "saccade_mean_mag":      saccade_mean_mag,
            "saccade_dir_var":       0 if pd.isna(saccade_dir_var) else saccade_dir_var,
            "saccade_circ_var":      saccade_circ_var,
            "blink_rate":            blink_rate,
            "valid_gaze_ratio":      valid_gaze_ratio,
        })

    return (gaze_df.dropna(subset=["win_id"])
                 .groupby(keys, dropna=False)
                 .apply(_agg)
                 .reset_index())

# 4) Merge fixation + gaze features (keep ON/OFF/phase separate)
def combine_fix_gaze_windows(fix_w: pd.DataFrame, gaze_w: pd.DataFrame) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]
    merged = pd.merge(fix_w, gaze_w, on=keys, how="outer", validate="one_to_one")
    return merged

In [48]:
# ---------- family helpers ----------
def _task_id_series(df: pd.DataFrame) -> pd.Series:
    if "block_id" in df.columns:
        # block_id format: "<video>|<user>|<taskid>"
        return df["block_id"].astype(str).str.split("|").str[-1].str.strip()
    raise KeyError("Need 'Task Id' or 'block_id' to determine family.")

def split_by_family(df: pd.DataFrame) -> dict[int, pd.DataFrame]:
    """Return {1: df_1x, 2: df_2x, 3: df_3x} keeping only ON/OFF rows per family."""
    tid = _task_id_series(df)
    out = {}
    for fam in (1, 2, 3):
        mask = tid.str.match(rf"^{fam}\.")
        d = df.loc[mask].copy()
        d["Task Id"] = tid.loc[mask]  # normalize column
        out[fam] = d
    return out

# ---------- 0) split your ON/OFF data FIRST ----------
fix_fams = split_by_family(fixations_onoff)   # dict {1: df, 2: df, 3: df}
gaz_fams = split_by_family(gaze_onoff)

# (optional) save the raw ON/OFF subsets per family for audit
for fam in (1, 2, 3):
    fix_fams[fam].to_csv(f"outputs/fix_onoff_task{fam}.csv", index=False)
    gaz_fams[fam].to_csv(f"outputs/gaze_onoff_task{fam}.csv", index=False)
    print(f"[family {fam}] fix rows={len(fix_fams[fam])}, gaze rows={len(gaz_fams[fam])}")

# ---------- 1) per-family windows + features (reuses your functions as-is) ----------
def make_family_windows(fix_df: pd.DataFrame,
                        gaze_df: pd.DataFrame,
                        *,
                        fam: int,
                        win_sec: float = 3.0) -> pd.DataFrame:
    if fix_df.empty and gaze_df.empty:
        print(f"[family {fam}] Empty split; skipping.")
        return pd.DataFrame()

    # shared anchor INSIDE the family split → aligns gaze & fix within that family only
    anchor = build_shared_anchor(fix_df, gaze_df)

    # win_id per family subset
    fix_wid  = add_window_id_by_timetick(fix_df,  win_sec=win_sec, shared_ref=anchor)
    gaze_wid = add_window_id_by_timetick(gaze_df, win_sec=win_sec, shared_ref=anchor)

    # features (cleaned)
    fix_w = window_features_fixations(fix_wid, win_sec=win_sec)   # <-- FIXED
    gaz_w = window_features_gaze(gaze_wid, win_sec=win_sec)

    # merge
    win = combine_fix_gaze_windows(fix_w, gaz_w)

    # order columns (only those present)
    cols_order = [
    "Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id",
    "fix_efficiency","fix_aoi_ratio",
    "gaze_dispersion_total",
    "pupil_mean","pupil_var",
    "saccade_mean_mag","saccade_dir_var","blink_rate","valid_gaze_ratio", "fix_rate_per_s","fix_median_dur","pupil_mad","pupil_slope","saccade_circ_var"
]
    win = win.reindex(columns=[c for c in cols_order if c in win.columns])

    # save
    out_path = f"outputs/windows_task{fam}.csv"
    win.to_csv(out_path, index=False)
    print(f"[family {fam}] windows saved: {out_path} (rows={len(win)})")
    return win

win_task1 = make_family_windows(fix_fams[1], gaz_fams[1], fam=1, win_sec=4.0)
win_task2 = make_family_windows(fix_fams[2], gaz_fams[2], fam=2, win_sec=4.0)
win_task3 = make_family_windows(fix_fams[3], gaz_fams[3], fam=3, win_sec=4.0)



[family 1] fix rows=26703, gaze rows=745183
[family 2] fix rows=7506, gaze rows=245632
[family 3] fix rows=6653, gaze rows=221766


/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_29833/4173348416.py:148: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_29833/4173348416.py:211: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)


[family 1] windows saved: outputs/windows_task1.csv (rows=3607)


/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_29833/4173348416.py:148: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_29833/4173348416.py:211: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)


[family 2] windows saved: outputs/windows_task2.csv (rows=1577)


/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_29833/4173348416.py:148: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)


[family 3] windows saved: outputs/windows_task3.csv (rows=1444)


/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_29833/4173348416.py:211: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)


In [44]:
# family 1 = Task 1.x
win_task1.to_csv("outputs/windows_task1.csv", index=False)

# family 2 = Task 2.x
win_task2.to_csv("outputs/windows_task2.csv", index=False)

# family 3 = Task 3.x
win_task3.to_csv("outputs/windows_task3.csv", index=False)

In [45]:
# --- your mappings ---
task_mapping = {
    '1.1': 'C011', '1.2': 'C100', '1.3': 'C000', '1.4': 'Cnull', '1.5': 'C100',
    '1.6': 'C110', '1.7': 'C011', '1.8': 'C001', '1.9': 'C111', '2.1': 'B000',
    '2.2': 'Bnull', '2.3': 'B112', '2.4': 'B113', '2.5': 'B012', '2.6': 'B013',
    '2.7': 'B002', '2.8': 'B102', '2.9': 'B103', '3.1': 'A000', '3.2': 'A100',
    '3.3': 'A010', '3.4': 'A101', '3.5': 'A001', '3.6': 'A111', '3.7': 'A110',
    '3.8': 'A011', '3.9': 'Anull'
}

animation_mapping = {
    'A': {
        'type': 'FADE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
    'B': {
        'type': 'BOUNCE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': {0: 'Left to Right', 1: 'Top to Bottom', 2: 'Bouncing', 3: 'Pulsing'},
    },
    'C': {
        'type': 'SCROLL',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
}

def decode_from_code(code: str):
    """
    Return dict with decoded fields from a code like 'C011', 'B112', or '*null'.
    """
    if not isinstance(code, str) or "null" in code.lower():
        return {
            "anim_family": np.nan,
            "anim_code": code if isinstance(code, str) else np.nan,
            "anim_type": "NO-ANIM",
            "anim_delay": np.nan,
            "anim_ease": np.nan,
            "anim_direction": np.nan,
            "block_anim": False,
        }

    fam = code[0]                 # 'A' / 'B' / 'C'
    spec = code[1:]               # e.g. '011' or '112'
    meta = animation_mapping.get(fam, {})

    # defaults
    delay = ease = direction = np.nan

    # parse indices safely
    try:
        i_delay = int(spec[0]) if len(spec) >= 1 else 0
        i_ease  = int(spec[1]) if len(spec) >= 2 else 0
        i_dir   = int(spec[2]) if len(spec) >= 3 else 0
    except ValueError:
        i_delay = i_ease = i_dir = 0

    # map delay/ease
    if "delay" in meta and isinstance(meta["delay"], list) and i_delay < len(meta["delay"]):
        delay = meta["delay"][i_delay]
    if "ease" in meta and isinstance(meta["ease"], list) and i_ease < len(meta["ease"]):
        ease = meta["ease"][i_ease]

    # map direction (B uses dict, A/C use list)
    if fam == "B" and isinstance(meta.get("direction"), dict):
        direction = meta["direction"].get(i_dir, np.nan)
    elif isinstance(meta.get("direction"), list) and i_dir < len(meta["direction"]):
        direction = meta["direction"][i_dir]

    return {
        "anim_family": fam,
        "anim_code": code,
        "anim_type": meta.get("type", np.nan),
        "delay": delay,
        "ease": ease,
        "direction": direction,
        "anim": True,
    }

def add_animation_columns(windows_df: pd.DataFrame) -> pd.DataFrame:
    """
    Make a copy of windows_df and add:
      anim_family, anim_code, anim_type, anim_delay, anim_ease, anim_direction, block_anim
    based on Task Id -> code mapping.
    """
    out = windows_df.copy()

    # ensure string task ids for mapping
    out["Task Id"] = out["Task Id"].astype(str).str.strip()

    # look up the code for each Task Id
    out["anim_code"] = out["Task Id"].map(task_mapping).fillna("Unknown")

    # decode each row
    decoded = out["anim_code"].apply(decode_from_code).apply(pd.Series)

    # attach
    for col in ["delay","ease","direction"]:
        out[col] = decoded[col]
    return out



In [46]:
windows_with_anim1 = add_animation_columns(win_task1)
windows_with_anim2 = add_animation_columns(win_task2)
windows_with_anim3 = add_animation_columns(win_task3)

windows_with_anim1.to_csv("outputs/windows_with_animation_details1.csv", index=False)
windows_with_anim2.to_csv("outputs/windows_with_animation_details2.csv", index=False)
windows_with_anim3.to_csv("outputs/windows_with_animation_details3.csv", index=False)
